# Tema: CDC y Change Data Feed

## Objetivos
Capturar INSERT/UPDATE/DELETE y distinguir eventos de cambio del estado actual.

## Conceptos importantes para el examen
CDC describe cambios de origen; Delta CDF expone cambios confirmados; _change_type, _commit_version y _commit_timestamp; preimage/postimage. CDF no es un archivo histórico permanente.

**Dificultad:** Intermedio · **Tiempo estimado:** 70 min.



Ejecuta la preparación una vez; después avanza celda a celda. Las soluciones modifican datos: úsalas tras tu intento. Para volver al estado inicial, ejecuta de nuevo la preparación completa (crea otro schema). No uses «Run all» para estudiar.

- [ ] Completado
- [ ] Necesito repasar
- [ ] Dominado

## Preparación y datos ficticios
Se necesita un notebook Python en Databricks con Spark y Unity Catalog. Solo se crean objetos en el schema de prácticas mostrado.

In [ ]:
# Cada ejecución de esta celda crea un schema NUEVO y aislado.
# El catálogo debe existir y permitir USE CATALOG y CREATE SCHEMA.
# Si no puedes crear schemas, pide uno de prácticas exclusivo y cambia SCHEMA.
import re
import uuid
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.window import Window

dbutils.widgets.text("catalog", spark.sql("SELECT current_catalog()").first()[0])
CATALOG = dbutils.widgets.get("catalog")
RUN_ID = uuid.uuid4().hex[:10]
SCHEMA = "dea_19_" + RUN_ID
def ident(value):
    return "`" + value.replace("`", "``") + "`"
spark.sql(f"USE CATALOG {ident(CATALOG)}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {ident(SCHEMA)}")
spark.sql(f"USE SCHEMA {ident(SCHEMA)}")
spark.sql("SET TIME ZONE 'UTC'")
print(f"Objetos de esta sesión: {CATALOG}.{SCHEMA}")
# No se borran automáticamente schemas, tablas ni checkpoints.


In [ ]:
customers = spark.createDataFrame(
    [(i, f"Cliente {i:02d}", ["Madrid", "Sevilla", "Bilbao"][i % 3],
      datetime(2026, 1, 1)) for i in range(1, 13)],
    "customer_id INT, name STRING, city STRING, updated_at TIMESTAMP"
)
customers.write.format("delta").mode("overwrite").saveAsTable("customers")
updates = spark.createDataFrame([
    (2, "Cliente 02", "Valencia", datetime(2026, 2, 1)),
    (5, "Cliente 05", "Zaragoza", datetime(2026, 2, 2)),
    (13, "Cliente 13", "Madrid", datetime(2026, 2, 3))],
    customers.schema)
updates.write.format("delta").mode("overwrite").saveAsTable("customers_updates")
display(customers)

In [ ]:
# Requiere CREATE VOLUME en el schema; alternativa: usa un volumen autorizado.
spark.sql("CREATE VOLUME IF NOT EXISTS lab_files")
BASE = f"/Volumes/{CATALOG}/{SCHEMA}/lab_files"
dbutils.fs.mkdirs(BASE + "/landing")
CHECKPOINT = BASE + "/checkpoints/main"
print(BASE)

## PARTE 1 - EJEMPLOS GUIADOS

### 1. Activar CDF antes de cambiar
La versión inicial existente no se convierte retroactivamente en cambios.

In [ ]:
spark.sql("ALTER TABLE customers SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")
CDF_START = spark.sql("DESCRIBE HISTORY customers").agg(F.max("version")).first()[0]
print("Inicio CDF:", CDF_START)

### 2. Producir cambios

In [ ]:
%sql
UPDATE customers SET city='Valencia', updated_at=TIMESTAMP '2026-02-01' WHERE customer_id=2;
DELETE FROM customers WHERE customer_id=3;
INSERT INTO customers VALUES (13, 'Cliente 13', 'Cádiz', TIMESTAMP '2026-02-02');

### 3. Leer eventos

In [ ]:
changes = spark.read.format("delta").option("readChangeFeed", "true").option("startingVersion", CDF_START).table("customers")
display(changes.orderBy("_commit_version", "customer_id", "_change_type"))

## PARTE 2 - EJERCICIOS
Resuelve todos antes de abrir las soluciones. Los ejercicios se realizan en orden y pueden usar resultados anteriores.

### EJERCICIO 1
Cuenta eventos por _change_type y localiza la imagen anterior y posterior del cliente 2.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


### EJERCICIO 2
Extrae solo inserts y update_postimage para un staging de upserts.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


### EJERCICIO 3
Consume CDF mediante streaming a una tabla de auditoría usando checkpoint propio.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


### EJERCICIO 4
Actualiza cliente 4 y reanuda la lectura. Verifica que se añaden dos eventos sin repetir los anteriores.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


### EJERCICIO 5
Ordena eventos útiles por commit y clave; explica por qué no basta la hora de llegada para eventos externos.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


## PARTE 3 - PISTAS
**Pista 1:** Una actualización produce dos imágenes.

**Pista 2:** Los deletes deben procesarse aparte, no ignorarse en una réplica completa.

**Pista 3:** readStream con readChangeFeed.

**Pista 4:** Conserva checkpoint y startingVersion.

**Pista 5:** En CDF el commit ordena versiones de esa tabla; un origen externo necesita su secuencia.

## PARTE 4 - SOLUCIONES
**Detente aquí si todavía estás practicando.** Referencias completas para comparar después de resolver. Puedes plegar esta sección en Databricks.

### Solución 1

In [ ]:
display(changes.groupBy("_change_type").count())
display(changes.filter("customer_id=2").orderBy("_commit_version", "_change_type"))

### Solución 2

In [ ]:
upserts = changes.filter("_change_type IN ('insert','update_postimage')")
upserts.write.format("delta").mode("overwrite").saveAsTable("cdc_upserts")
changes.filter("_change_type='delete'").write.format("delta").mode("overwrite").saveAsTable("cdc_deletes")

### Solución 3

In [ ]:
def run_cdf():
    q = (spark.readStream.option("readChangeFeed","true").option("startingVersion", CDF_START).table("customers")
      .writeStream.format("delta").option("checkpointLocation",CHECKPOINT)
      .trigger(availableNow=True).toTable("cdc_audit"))
    q.awaitTermination()
run_cdf()
assert spark.table("cdc_audit").count() == 4

### Solución 4

In [ ]:
spark.sql("UPDATE customers SET city='Lugo', updated_at=TIMESTAMP '2026-03-01' WHERE customer_id=4")
run_cdf()
assert spark.table("cdc_audit").count() == 6

### Solución 5

In [ ]:
display(spark.table("cdc_audit").filter("_change_type != 'update_preimage'").orderBy("_commit_version","customer_id"))
# _commit_version ordena commits de esta tabla, no de todas las tablas.
# Para CDC externo usa LSN/offset o (updated_at,event_id) con orden definido.
# SCD decide cómo representar esos cambios en la dimensión; CDC los transporta.

## PARTE 5 - PREGUNTAS TIPO EXAMEN
Preguntas originales de práctica; no son preguntas oficiales.

### Pregunta 1
¿Qué imagen usarías para actualizar el estado de un cliente?

A. update_preimage

B. update_postimage

C. Todas sin distinguir

D. Solo timestamp

### Pregunta 2
¿Qué ocurre con cambios anteriores a activar CDF?

A. Siempre se reconstruyen automáticamente

B. Se inventan desde la fecha actual

C. No quedan capturados retroactivamente

D. Se convierten en DELETE

### Pregunta 3
¿Qué diferencia CDC de SCD?

A. CDC comunica cambios; SCD define su representación en una dimensión

B. Son siempre lo mismo

C. SCD es solo un protocolo de red

D. CDC exige SCD2

### Respuestas y explicación
**1. B** — Contiene los valores posteriores.

**2. C** — CDF empieza a registrar desde su activación.

**3. A** — Puedes consumir CDC para mantener SCD1 o SCD2.

## PARTE 6 - RETO FINAL
Crea una réplica desde un snapshot inicial y aplica CDF posterior con altas, modificaciones y bajas. Comprueba igualdad final y explica cómo evitarías perder cambios entre snapshot y consumo.

Anota tu decisión, implementa el código y muestra evidencias. No se incluye solución para este reto.

In [ ]:
# TU RETO: código y verificaciones
